# PyBullet 慢在哪里 —— 一个可以自己跑的对照实验

`pip install pybullet` 在 Linux 上装出来的 PyBullet，`getCameraImage` 比它应有的速度慢一个数量级。
原因不在渲染，在于**返回值的编组方式**。这个 notebook 让你自己测一遍。

结论预告（A100，1280×720，Franka Panda）：

| | numpy off | numpy on |
|---|---|---|
| depth only | 282 ms | **19 ms** |
| depth + segmentation | 261 ms | **25 ms** |

---

### 跑之前请先看这两条

**1. 会编译两次 PyBullet，时间取决于核数。**
PyPI 没有 Linux wheel，每次装 PyBullet 都是现场编译 —— 这本身就是故事的一部分。
96 核机器上实测每次约 1 分钟；Colab 只有 2 个核，一次十几分钟是正常的。
编译输出是实时流式打印的，所以你能看到它在动 —— 长时间没有新输出才是异常。

**2. 不会碰你的环境。**
两个版本都用 `pip install --target` 装到 `data/cache/pybullet_bench/` 下面，
venv / Colab 运行时里已有的 PyBullet 原样不动。benchmark 在独立子进程里跑
（PyBullet 是编译好的 `.so`，一个进程里没法同时载入两个版本）。

GPU 不是必需的。后面会看到，**有没有 GPU 对结论没有影响**，这恰恰是关键证据。


## 0. 环境


In [ ]:
import importlib.util
import os
import sys
import tempfile

IN_COLAB = (
  importlib.util.find_spec("google") is not None
  and importlib.util.find_spec("google.colab") is not None
)


def find_repo(start):
  d = os.path.abspath(start)
  while not os.path.exists(os.path.join(d, 'core', 'physics.py')):
    parent = os.path.dirname(d)
    if parent == d:
      return None
    d = parent
  return d


REPO_DIR = find_repo(os.getcwd())

if REPO_DIR:
  WORK = os.path.join(REPO_DIR, 'data', 'cache', 'pybullet_bench')
elif IN_COLAB:
  WORK = '/content/pybullet_bench'
else:
  WORK = os.path.join(tempfile.gettempdir(), 'pybullet_bench')

PB_OFF = os.path.join(WORK, 'pb_numpy_off')
PB_ON = os.path.join(WORK, 'pb_numpy_on')
os.makedirs(WORK, exist_ok=True)

print(f"{'Colab' if IN_COLAB else 'Local'}  |  python {sys.version.split()[0]}")
print(f'work dir : {WORK}')

## 1. 编译两个版本

唯一的区别是一个 flag：

| | 构建方式 | 结果 |
|---|---|---|
| **numpy off** | pip 默认的**隔离构建环境** | 那个环境里没有 numpy |
| **numpy on** | `--no-build-isolation` | 构建过程能 import 到 numpy |

PyBullet 的 `setup.py` 用一句裸的 `try: import numpy` 来探测，探测失败就静默降级。
它其实打印了 `numpy is disabled. getCameraImage maybe slower.`，但 pip 默认把构建输出藏起来。

`--no-cache-dir` 也是必需的，否则 pip 会直接复用前一次编好的 wheel，第二次构建等于没做。

> 下面这个 cell 是最慢的一步。输出是实时的，看到 `Building wheel for pybullet` 就是正常在编。


In [ ]:
import subprocess
import time


def build(target, isolated):
  if os.path.exists(os.path.join(target, 'pybullet_data')):
    print(f'[SKIP] {os.path.basename(target)} already built')
    return
  cmd = [
    sys.executable,
    '-m',
    'pip',
    'install',
    '--target',
    target,
    '--no-binary',
    'pybullet',
    '--no-cache-dir',
    'pybullet',
  ]
  if not isolated:
    cmd.insert(-1, '--no-build-isolation')
  print(f"$ {' '.join(cmd)}")
  t0 = time.time()
  proc = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
  )
  for line in proc.stdout:
    if any(
      k in line
      for k in ('numpy is', 'Building wheel', 'Created wheel', 'Successfully', 'ERROR', 'error:')
    ):
      print('   ', line.rstrip()[:110])
  proc.wait()
  print(f'    -> exit {proc.returncode} in {time.time() - t0:.0f}s\n')


build(PB_OFF, isolated=True)
build(PB_ON, isolated=False)

注意两次构建的 `numpy is ...` 那一行 —— 这就是全部的区别。


In [ ]:
import subprocess


def probe_flag(target):
  out = subprocess.run(
    [sys.executable, '-c', 'import pybullet as p; print(p.isNumpyEnabled())'],
    env={**os.environ, 'PYTHONPATH': target},
    capture_output=True,
    text=True,
  )
  return out.stdout.strip().splitlines()[-1]


print('numpy off build -> isNumpyEnabled() =', probe_flag(PB_OFF))
print('numpy on  build -> isNumpyEnabled() =', probe_flag(PB_ON))

这个开关是什么意思，PyBullet 自己的文档写得很清楚：


In [ ]:
out = subprocess.run(
  [sys.executable, '-c', 'import pybullet as p; print(p.isNumpyEnabled.__doc__)'],
  env={**os.environ, 'PYTHONPATH': PB_ON},
  capture_output=True,
  text=True,
)
print(out.stdout.strip().splitlines()[-1])

## 2. 跑同一个 benchmark

两个构建跑的是**同一个脚本**，只有 `PYTHONPATH` 不同。


In [ ]:
BENCH_PY = os.path.join(WORK, 'bench.py')

bench_src = r'''
import importlib.util, json, sys, time
import numpy as np
import pybullet as p
import pybullet_data

out_json, out_depth = sys.argv[1], sys.argv[2]

p.connect(p.DIRECT)
p.setAdditionalSearchPath(pybullet_data.getDataPath())

# Try the GPU rasteriser. The module pybullet ships is "eglRenderer";
# "_eglRendererPlugin" is only the name it registers under.
egl = False
spec = importlib.util.find_spec("eglRenderer")
if spec is not None:
    egl = p.loadPlugin(spec.origin, "_eglRendererPlugin") >= 0

robot = p.loadURDF("franka_panda/panda.urdf", useFixedBase=True)
for i, a in enumerate([0.0, -0.3, 0.0, -2.0, 0.0, 1.8, 0.8]):
    p.resetJointState(robot, i, a)

W, H = 1280, 720
view = p.computeViewMatrix([1.2, 0.0, 0.6], [0.0, 0.0, 0.3], [0.0, 0.0, 1.0])
proj = p.computeProjectionMatrixFOV(60, W / H, 0.01, 10)

def shoot(flags):
    return p.getCameraImage(W, H, viewMatrix=view, projectionMatrix=proj,
                            renderer=p.ER_BULLET_HARDWARE_OPENGL, flags=flags)

def bench(flags, n=8):
    shoot(flags)                       # warm up
    t0 = time.perf_counter()
    for _ in range(n):
        shoot(flags)
    return (time.perf_counter() - t0) / n * 1000.0

frame = shoot(p.ER_NO_SEGMENTATION_MASK)
result = {
    "numpy_enabled": int(p.isNumpyEnabled()),
    "egl": bool(egl),
    "rgb_type": type(frame[2]).__name__,
    "depth_type": type(frame[3]).__name__,
    "depth_only_ms": bench(p.ER_NO_SEGMENTATION_MASK),
    "with_seg_ms": bench(p.ER_SEGMENTATION_MASK_OBJECT_AND_LINKINDEX),
}

# Exactly what core/physics.py:render_depth does -- no dtype coercion, so the
# tuple path stays float64 and the ndarray path stays float32, and the two
# builds are compared on the number the pipeline actually consumes.
buf = np.reshape(frame[3], (H, W))
metric = 0.1 / (10.0 - 9.99 * buf)
np.save(out_depth, np.where(metric < 9.9, metric, 0.0))
json.dump(result, open(out_json, "w"))
print(json.dumps(result, indent=2))
'''
open(BENCH_PY, 'w').write(bench_src)
print('written ->', BENCH_PY)

In [ ]:
def run_bench(target, tag):
  print(f'--- {tag} ---')
  out = subprocess.run(
    [
      sys.executable,
      BENCH_PY,
      os.path.join(WORK, f'{tag}.json'),
      os.path.join(WORK, f'{tag}_depth.npy'),
    ],
    env={**os.environ, 'PYTHONPATH': target},
    capture_output=True,
    text=True,
  )
  print(out.stdout.strip() or out.stderr.strip()[-2000:])


run_bench(PB_OFF, 'off')
run_bench(PB_ON, 'on')

## 3. 关键的一行

看 `rgb_type` 和 `depth_type`：**`tuple` → `ndarray`**。

一张 1280×720 的 RGBA 图，numpy off 的 PyBullet 把它装成一个
**3,686,400 个元素的 Python tuple** 再返回 —— 每个像素通道都是一个装箱的 Python int。
深度图是另一个 921,600 元素的 tuple。

光是构造这么大的 tuple 要花多久？不需要 PyBullet 就能量：


In [ ]:
import time

t0 = time.perf_counter()
rgb_sized = tuple(range(1280 * 720 * 4))
print(f'构造一个 3,686,400 元素的 tuple: {(time.perf_counter() - t0) * 1000:.0f} ms')

del rgb_sized

这就是那 ~280 ms 的主要去向 —— 而且 **RGB 那部分你多半根本不用**
（做机器人 mask / depth 的流水线只要 depth 和 segmentation），PyBullet 也没有任何 flag 能关掉它。

## 4. 对比


In [ ]:
import json

off = json.load(open(os.path.join(WORK, 'off.json')))
on = json.load(open(os.path.join(WORK, 'on.json')))

print(f"GPU (EGL) 可用: {off['egl']}\n")
print(f"{'':24s}{'numpy off':>12s}{'numpy on':>12s}{'加速':>9s}")
for key, label in [('depth_only_ms', 'depth only'), ('with_seg_ms', 'depth + segmentation')]:
  a, b = off[key], on[key]
  print(f'{label:24s}{a:>10.1f}ms{b:>10.1f}ms{a / b:>8.1f}x')

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

off = json.load(open(os.path.join(WORK, 'off.json')))
on = json.load(open(os.path.join(WORK, 'on.json')))

C_OFF, C_ON = '#eb6834', '#2a78d6'
SURFACE, INK, MUTED, GRID = '#fcfcfb', '#0b0b0b', '#52514e', '#e5e4e0'

rows = [('depth only', 'depth_only_ms'), ('depth +\nsegmentation', 'with_seg_ms')]
y = np.arange(len(rows)) * 1.0
h, gap = 0.26, 0.03
xmax = max(off['depth_only_ms'], off['with_seg_ms'])

fig, ax = plt.subplots(figsize=(8.6, 3.4), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

for offset, colour, label, src in [
  (-(h / 2 + gap), C_ON, 'numpy on  (rebuilt)', on),
  (+(h / 2 + gap), C_OFF, 'numpy off  (pip default)', off),
]:
  vals = [src[k] for _, k in rows]
  bars = ax.barh(y + offset, vals, height=h, color=colour, label=label, zorder=3)
  for bar, v in zip(bars, vals):
    ax.text(
      v + xmax * 0.012,
      bar.get_y() + bar.get_height() / 2,
      f'{v:.0f} ms',
      va='center',
      ha='left',
      fontsize=10,
      color=INK,
      zorder=4,
    )

ax.set_yticks(y, [lbl for lbl, _ in rows], fontsize=10, color=INK)
ax.invert_yaxis()
ax.set_ylim(len(rows) - 0.5, -0.62)
ax.set_xlim(0, xmax * 1.16)
ax.set_xlabel('getCameraImage, ms per call (lower is better)', fontsize=10, color=MUTED)
ax.tick_params(axis='x', colors=MUTED, labelsize=9, length=0)
ax.tick_params(axis='y', length=0)
ax.grid(axis='x', color=GRID, linewidth=1, zorder=0)
ax.set_axisbelow(True)
for side in ('top', 'right', 'left'):
  ax.spines[side].set_visible(False)
ax.spines['bottom'].set_color('#d8d7d2')

leg = ax.legend(
  loc='lower left',
  bbox_to_anchor=(-0.005, 1.005),
  ncol=2,
  frameon=False,
  fontsize=9.5,
  handlelength=0.9,
  handleheight=0.9,
  columnspacing=1.6,
)
for text in leg.get_texts():
  text.set_color(MUTED)

ax.set_title(
  'PyBullet getCameraImage  |  1280x720, Franka Panda', fontsize=11.5, color=INK, loc='left', pad=34
)
plt.tight_layout()
plt.show()

## 5. 但是结果一样吗？

加速如果改变了输出，那就不是加速而是行为变更。这里比的是 `core/physics.py` 里
`render_depth` 实际消费的那张米制深度图。


In [ ]:
import numpy as np

d_off = np.load(os.path.join(WORK, 'off_depth.npy'))
d_on = np.load(os.path.join(WORK, 'on_depth.npy'))

print(f'dtype      : {d_off.dtype} (tuple 路径)  ->  {d_on.dtype} (ndarray 路径)')

diff = np.abs(d_off.astype(np.float64) - d_on.astype(np.float64))
print(f'逐位相同   : {np.array_equal(d_off, d_on)}')
print(f'最大差异   : {diff.max():.3e} m')
print('流水线阈值 : 1.5e-02 m (自遮挡) / 2.0e-02 m (环境遮挡)')
if diff.max() > 0:
  print(f'余量       : {1.5e-2 / diff.max():.0f}x')

差异来自 float32 与 Python float64 在 `0.1 / (10 - 9.99*d)` 这步的舍入，量级在微米。
分割掩码和 link id 是整数，逐位相同。

## 6. 最关键的一条证据：GPU 根本不重要

如果瓶颈真在光栅化，那么 GPU 光栅化器应该远快于 CPU 光栅化器。
同一个构建，只换 `renderer` 参数：


In [ ]:
PROBE_PY = os.path.join(WORK, 'probe.py')

probe_src = r'''
import importlib.util, time
import pybullet as p
import pybullet_data

p.connect(p.DIRECT)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
spec = importlib.util.find_spec("eglRenderer")
if spec is not None:
    p.loadPlugin(spec.origin, "_eglRendererPlugin")
p.loadURDF("franka_panda/panda.urdf", useFixedBase=True)
view = p.computeViewMatrix([1.2, 0.0, 0.6], [0.0, 0.0, 0.3], [0.0, 0.0, 1.0])
proj = p.computeProjectionMatrixFOV(60, 16 / 9, 0.01, 10)

def t(renderer, n=6):
    p.getCameraImage(1280, 720, viewMatrix=view, projectionMatrix=proj,
                     renderer=renderer)
    t0 = time.perf_counter()
    for _ in range(n):
        p.getCameraImage(1280, 720, viewMatrix=view, projectionMatrix=proj,
                         renderer=renderer)
    return (time.perf_counter() - t0) / n * 1000.0

print("  numpy=%d   GPU/EGL %7.1f ms    CPU/TinyRenderer %7.1f ms"
      % (p.isNumpyEnabled(), t(p.ER_BULLET_HARDWARE_OPENGL), t(p.ER_TINY_RENDERER)))
'''
open(PROBE_PY, 'w').write(probe_src)

for target, tag in [(PB_OFF, 'numpy off'), (PB_ON, 'numpy on ')]:
  out = subprocess.run(
    [sys.executable, PROBE_PY],
    env={**os.environ, 'PYTHONPATH': target},
    capture_output=True,
    text=True,
  )
  line = [l for l in out.stdout.splitlines() if 'numpy=' in l]
  print(f'{tag} :', line[0].strip() if line else out.stderr.strip()[-300:])

**GPU 和 CPU 光栅化器在两种构建下都几乎打平。**

如果渲染是瓶颈，这不可能发生。两边真正在花时间的是同一件事 —— 把像素搬进 Python 对象。
修好之后两者才双双降到几十毫秒。

（顺带：即使开了 numpy，GPU 仍未明显胜出，说明剩下的开销是 buffer 回读而不是光栅化。
这才是 nvdiffrast 这类方案真正能进一步压缩的部分 —— 但对标的是这 25 ms，不是原来的 280 ms。）

## 怎么用到自己的项目里

```bash
pip install --force-reinstall --no-deps --no-binary pybullet \
    --no-build-isolation --no-cache-dir pybullet
```

然后断言一下，别让它悄悄退回去：

```python
import pybullet
assert pybullet.isNumpyEnabled(), 'pybullet 没有 numpy 支持，getCameraImage 会慢十倍'
```

本仓库里这一步已经加进 `setup.sh`，并用 `isNumpyEnabled()` 做了守卫，重复运行不会重新编译。

跑完想回收空间的话：`rm -rf data/cache/pybullet_bench`（两个构建各 256 MB，共约 512 MB）。
